# Finding carcinogenic, mutagenic, or reprotoxic (CMR) substances in plastics
This notebook demonstrates how to screen for CMR substances that may be associated with plastics. Plastic-associated substances are identified using the PlastChem database, while CMR status is inferred from CMR-related GHS classifications published under Annex VI of the EU CLP Regulation.

**Prerequisites**
- Download a copy of the [example global inventory](https://github.com/envris/global-inventory-compiler/tree/main/data) from the GIC repository.
- Download the PlastChem database from the [PlastChem website](https://plastchem-project.org/).
- Prepare a DataFrame of GHS codes using the CLP extraction example notebook.

## Step 1: Load the inventory
Create an Inventory instance from the precompiled global inventory file.

In [ ]:
from gic import Inventory

inv = Inventory("input/global_inventory.jsonl")

## Step 2: Extract the PlastChem database into a DataFrame
This example extracts only PubChem CIDs, PubChem names, and IUPAC names. CAS RNs are not included in the public GIC release and are therefore not used here. Rows that contain no usable identifiers are logged as extraction errors.

Many PlastChem records also fall outside the scope of the global inventory because they represent contaminants, degradation products, or impurities rather than inventory-listed substances. As a result, only a subset of the full PlastChem database is suitable for this analysis.

Although the database includes GHS information, that information is not extracted here because the hazard classifications are sourced directly from the original CLP reference data.

In [ ]:
from gic import GenericXlsxExtractor

# Extract identifiers from the PlastChem database.
extractor = GenericXlsxExtractor()
extractor.input_file = "input/plastchem_db_v1.0.xlsx"
extractor.sheet_name = "Full database"
extractor.output_file = "output/plastchem.jsonl"
extractor.errors_file = "output/plastchem_errors.tsv"
extractor.columns = {
    5: "cid",
    6: "chemical_name",
    7: "chemical_name",
}  # PubChem CID, PubChem name, and IUPAC name.
extractor.constants = {"list_tag": ["PlastChem"]}
extractor.start_row = 3

plastchem_df = extractor.extract()

# Assess how well the PlastChem entries map to the inventory.
mapped = 0
unmappable = 0
out_of_scope = 0
inv.fill(plastchem_df, "sid")  # Copy SIDs from the inventory into the data frame
for row in plastchem_df:
    if row.get_values("sid"):
        mapped += 1
    elif not row.get_values("cid"):
        unmappable += 1
    else:
        out_of_scope += 1

print(f"Number of rows extracted from the PlastChem database: {len(plastchem_df)}")
print(f"Number of unmappable rows without PubChem CID: {unmappable}")
print(f"Number of mappable rows outside inventory scope: {out_of_scope}")
print(plastchem_df[4].as_dict())

Number of rows extracted from the PlastChem Database: 12955
Number of unmappable rows (without PubChem CID): 0
Number of mappable rows outside of global inventory scope: 2581
{'cid': ['CID7488'], 'chemical_name': ['Terephthaloyl chloride', 'benzene-1,4-dicarbonyl chloride'], 'list_tag': ['PlastChem'], 'sid': [2742]}


## Step 3: Import the PlastChem data into the inventory
Importing the PlastChem data provides plastic-related context for substances already represented in the inventory. In this example, the import is performed as an auxiliary list so that only the *PlastChem* list tag is added, without creating or merging substances.

In [ ]:
inv.import_from_data_frame(plastchem_df, auxiliary=True)
plastchem_sids = inv.get_by_list_tags("sid", ["PlastChem"])
print(
    "Number of plastic-associated substances in the global inventory: "
    f"{len(plastchem_sids)}"
)

Number of plastic-associated substances in the global inventory: 11933


## Step 4: Identify substances with CMR-related GHS classifications
Load the GHS DataFrame and map it to the inventory so that the SIDs associated with CMR-relevant GHS codes can be identified.

In [ ]:
from gic import DataFrame

ghs_df = DataFrame.from_jsonl("output/eu_clp_ghs.jsonl")
inv.fill(ghs_df, "sid")  # Populate GHS Dataframe with sids

# Define CMR GHS codes
cmr_ghs_codes = {
    "H340",  # Mutagenic Cat. 1A, 1B
    "H341",  # Mutagenic Cat. 2
    "H350",  # Carcinogenic Cat. 1A, 1B
    "H351",  # Carcinogenic Cat. 2
    "H360",  # Reproductive Toxic Cat. 1A, 1B
    "H360F",  # Reproductive Toxic Cat. 1A, 1B
    "H360D",  # Reproductive Toxic Cat. 1A, 1B
    "H360FD",  # Reproductive Toxic Cat. 1A, 1B
    "H360Fd",  # Reproductive Toxic Cat. 1A, 1B
    "H360Df",  # Reproductive Toxic Cat. 1A, 1B
    "H361",  # Reproductive Toxic Cat. 2
    "H361f",  # Reproductive Toxic Cat. 2
    "H361d",  # Reproductive Toxic Cat. 2
    "H361fd",  # Reproductive Toxic Cat. 2
}

# Get sids associated with CMR GHS codes
cmr_sids = set()
for row in ghs_df:
    for ghs_code in row.get_values("ghs_code"):
        if ghs_code in cmr_ghs_codes:
            cmr_sids.update(row.get_values("sid"))

# Index DataFrame for fast lookup by SID (used later)
ghs_df.index_by_identifier("sid")

print(f"Number of substances with CMR GHS classifications: {len(cmr_sids)}")

Number of substances with CMR GHS classifications: 1472


## Step 5: Export plastic-associated substances with CMR classifications
Intersect the PlastChem-derived SIDs with the CMR-associated SIDs, then export the matching substances together with their identifiers, regulatory context, and relevant GHS codes.

In [ ]:
cmr_plastchem_sids = sorted(list(set(plastchem_sids).intersection(cmr_sids)))
print(f"Number of CMR substances in PlastChem Database: {len(cmr_plastchem_sids)}")

# Extract substance data from inventory
results = inv.export_as_data_frame(cmr_plastchem_sids)

# Import relevant GHS information into results
for row in results:
    sids = row.get_values("sid")
    ghs_codes = ghs_df.get_values_by_identifiers("ghs_code", sids)
    ghs_codes = list(set(ghs_codes).intersection(cmr_ghs_codes))
    row.import_from_dict({"ghs_code": ghs_codes})

# Save to Excel file
results.to_xlsx("output/cmr_plastic_substances.xlsx")

Number of CMR substances in PlastChem Database: 422
